# 07 — Sentence Trajectories

**Engineering question**

How do sentence tokens trace paths through transformer representation space?

Notebook 00 defined the conceptual map:

```text
sentence tokens
→ hidden states
→ sentence trajectory
→ curvature measurement
→ prediction geometry
```

This notebook begins the real measurement pipeline.

It loads a pretrained autoregressive transformer, extracts hidden states for a small sentence batch, and visualizes token trajectories across layers.

Curvature itself is reserved for Notebook 13. Here the goal is to establish the object being measured:

\[
x_i^\ell
\]

the hidden representation of token \(i\) at transformer layer \(\ell\).


## Setup

This notebook is designed to run from either:

- the repo root, or
- inside `notebooks/`.

It creates:

```text
figures/
results/csv/
results/json/
results/arrays/
results/07_outputs.zip
```

If `transformers` is unavailable, install the repo requirements first:

```bash
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import zipfile
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"
ARR = RES / "arrays"

for p in [FIG, RES, CSV, JS, ARR]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Load model and tokenizer

Use a small GPT-style model first.

The aim is not model quality. The aim is to validate the trajectory extraction pipeline quickly.

Default:

```text
distilgpt2
```

You can switch to `gpt2` after the pipeline works.


In [ ]:
MODEL_NAME = "distilgpt2"

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_hidden_states=True)
    model.to(DEVICE)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("model:", MODEL_NAME)
    print("device:", DEVICE)
    print("layers:", model.config.n_layer if hasattr(model.config, "n_layer") else "unknown")
    print("hidden size:", model.config.n_embd if hasattr(model.config, "n_embd") else "unknown")
except Exception as e:
    print("Model setup failed.")
    print("Install dependencies with: pip install -r requirements.txt")
    raise e


## 2. Sentence batch

Use a small batch of sentences that directly connects to the repo.

The batch should include:

- the lab report title,
- a plain sentence,
- a paper-like sentence,
- a geometry sentence,
- a curvature sentence.

This gives enough variation to check that the extraction code is not overfit to one example.


In [ ]:
sentences = [
    "Trajectory straightening specifies language prediction.",
    "The cat sat on the mat.",
    "Large language models predict the next token.",
    "Geometry organizes predictive representations.",
    "Curvature decreases across transformer depth.",
]

batch = tokenizer(
    sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

batch = {k: v.to(DEVICE) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch, output_hidden_states=True)

hidden_states = outputs.hidden_states
# Tuple length = embeddings + transformer layers.
# Each entry shape = [batch, sequence, hidden_dim].
hidden = torch.stack(hidden_states, dim=0).detach().cpu().numpy()

input_ids = batch["input_ids"].detach().cpu().numpy()
attention_mask = batch["attention_mask"].detach().cpu().numpy()

tokens_by_sentence = []
for row in input_ids:
    tokens_by_sentence.append(tokenizer.convert_ids_to_tokens(row))

print("hidden tensor shape:", hidden.shape)
print("meaning: [layers including embeddings, batch, tokens, hidden_dim]")
print("input_ids shape:", input_ids.shape)


## 3. Hidden-state tensor shape

The core object is a tensor:

```text
layers × sentences × tokens × hidden_dimension
```

This is the data structure every later notebook will reuse.


In [ ]:
shape_record = {
    "model": MODEL_NAME,
    "hidden_shape": list(hidden.shape),
    "input_ids_shape": list(input_ids.shape),
    "num_hidden_layers_including_embedding": int(hidden.shape[0]),
    "batch_size": int(hidden.shape[1]),
    "sequence_length": int(hidden.shape[2]),
    "hidden_dim": int(hidden.shape[3]),
}

with open(JS / "07_hidden_state_shape.json", "w") as f:
    json.dump(shape_record, f, indent=2)

shape_df = pd.DataFrame(
    [
        {"axis": 0, "name": "layers including embedding", "size": hidden.shape[0]},
        {"axis": 1, "name": "sentences", "size": hidden.shape[1]},
        {"axis": 2, "name": "tokens", "size": hidden.shape[2]},
        {"axis": 3, "name": "hidden dimension", "size": hidden.shape[3]},
    ]
)

shape_df.to_csv(CSV / "07_hidden_state_shape.csv", index=False)
shape_df


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))

labels = shape_df["name"].tolist()
sizes = shape_df["size"].tolist()
x = np.arange(len(labels))

ax.bar(x, sizes)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("size")
ax.set_title("Hidden-State Tensor Shape")
ax.grid(True, axis="y", alpha=0.25)

for xi, size in zip(x, sizes):
    ax.text(xi, size, str(size), ha="center", va="bottom", fontsize=10)

savefig(fig, "07_hidden_state_shapes.png")
plt.show()


## 4. Token metadata

Store the tokenized batch.

This matters because the paper treats tokens as trajectory positions. The \(i\)-th token has hidden state \(x_i^\ell\).


In [ ]:
rows = []
for s_idx, sentence in enumerate(sentences):
    for t_idx, token in enumerate(tokens_by_sentence[s_idx]):
        active = int(attention_mask[s_idx, t_idx])
        rows.append(
            {
                "sentence_index": s_idx,
                "sentence": sentence,
                "token_index": t_idx,
                "token": token,
                "active": active,
            }
        )

token_df = pd.DataFrame(rows)
token_df.to_csv(CSV / "07_sentence_metadata.csv", index=False)
token_df.head(30)


In [ ]:
# Token count per sentence, excluding padding.
counts = token_df[token_df["active"] == 1].groupby("sentence_index").size()

fig, ax = plt.subplots(figsize=(8.8, 4.2))
ax.bar(counts.index.astype(str), counts.values)
ax.set_title("Active Token Count Per Sentence")
ax.set_xlabel("sentence index")
ax.set_ylabel("active tokens")
ax.grid(True, axis="y", alpha=0.25)

for xi, val in enumerate(counts.values):
    ax.text(xi, val, str(int(val)), ha="center", va="bottom")

savefig(fig, "07_token_counts.png")
plt.show()


## 5. PCA helper

To visualize trajectories, project high-dimensional hidden states to 2D.

For Notebook 07, PCA is only a visualization tool.

The actual curvature notebook should compute geometry in the original hidden-state space, not in PCA space.


In [ ]:
from sklearn.decomposition import PCA

def active_hidden_for_sentence(sentence_index, layer_index):
    mask = attention_mask[sentence_index].astype(bool)
    return hidden[layer_index, sentence_index, mask, :]

def pca_project(points, n_components=2):
    if points.shape[0] < 2:
        raise ValueError("Need at least two tokens to project a trajectory.")
    pca = PCA(n_components=n_components)
    xy = pca.fit_transform(points)
    return xy, pca

def clean_token(token):
    return token.replace("Ġ", " ").replace("Ċ", "\\n")


## 6. One sentence trajectory across selected layers

Start with the lab-report sentence:

```text
Trajectory straightening specifies language prediction.
```

Plot its hidden-state trajectory for several layers.


In [ ]:
sentence_index = 0
selected_layers = [0, 1, 2, 4, 6]

max_layer = hidden.shape[0] - 1
selected_layers = [min(layer, max_layer) for layer in selected_layers]

fig, axes = plt.subplots(1, len(selected_layers), figsize=(4.0 * len(selected_layers), 3.8))

if len(selected_layers) == 1:
    axes = [axes]

for ax, layer_index in zip(axes, selected_layers):
    points = active_hidden_for_sentence(sentence_index, layer_index)
    xy, pca = pca_project(points)

    ax.plot(xy[:, 0], xy[:, 1], marker="o", linewidth=2.0)
    ax.scatter(xy[0, 0], xy[0, 1], s=80, marker="s", label="start")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=80, marker="*", label="end")

    ax.set_title(f"Layer {layer_index}\\nPCA var {pca.explained_variance_ratio_.sum():.2f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.25)

axes[0].legend(loc="best")
fig.suptitle("Sentence Trajectory Across Selected Layers", fontsize=16, y=1.03)

savefig(fig, "07_sentence_trajectory_selected_layers.png")
plt.show()


## 7. Layer projection grid

This grid is the main visual output of Notebook 07.

It shows how a single sentence trajectory changes across transformer depth.

The grid is not yet a curvature plot. It is a trajectory extraction sanity check.


In [ ]:
sentence_index = 0
num_layers_total = hidden.shape[0]

grid_layers = np.linspace(0, num_layers_total - 1, min(num_layers_total, 8), dtype=int)
ncols = 4
nrows = math.ceil(len(grid_layers) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.6 * nrows))
axes = np.array(axes).reshape(-1)

for ax, layer_index in zip(axes, grid_layers):
    points = active_hidden_for_sentence(sentence_index, layer_index)
    xy, pca = pca_project(points)

    ax.plot(xy[:, 0], xy[:, 1], marker="o", linewidth=1.8)
    ax.scatter(xy[0, 0], xy[0, 1], s=65, marker="s")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=90, marker="*")
    ax.set_title(f"Layer {layer_index}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(True, alpha=0.2)

for ax in axes[len(grid_layers):]:
    ax.axis("off")

fig.suptitle("Layerwise Token Trajectory Projection", fontsize=16, y=1.02)

savefig(fig, "07_layer_projection_grid.png")
plt.show()


## 8. Trajectory statistics

Before measuring curvature, compute simple descriptive quantities:

- active token count,
- trajectory length in hidden-state space,
- average step length,
- PCA variance explained for a 2D projection.

These are not the paper's curvature metric. They are sanity checks that trajectories are being extracted consistently.


In [ ]:
def trajectory_step_lengths(points):
    diffs = np.diff(points, axis=0)
    return np.linalg.norm(diffs, axis=1)

records = []
for s_idx, sentence in enumerate(sentences):
    for layer_index in range(hidden.shape[0]):
        points = active_hidden_for_sentence(s_idx, layer_index)
        steps = trajectory_step_lengths(points)

        xy, pca = pca_project(points)

        records.append(
            {
                "sentence_index": s_idx,
                "sentence": sentence,
                "layer": layer_index,
                "active_tokens": int(points.shape[0]),
                "trajectory_length": float(steps.sum()),
                "mean_step_length": float(steps.mean()) if len(steps) else np.nan,
                "pca_2d_variance": float(pca.explained_variance_ratio_.sum()),
            }
        )

stats = pd.DataFrame(records)
stats.to_csv(CSV / "07_layer_statistics.csv", index=False)
stats.to_json(JS / "07_layer_statistics.json", orient="records", indent=2)

stats.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8))

for s_idx, sentence in enumerate(sentences):
    subset = stats[stats["sentence_index"] == s_idx]
    ax.plot(
        subset["layer"],
        subset["trajectory_length"],
        marker="o",
        markersize=3,
        label=f"sentence {s_idx}",
    )

ax.set_title("Trajectory Length Across Layers")
ax.set_xlabel("layer")
ax.set_ylabel("trajectory length in hidden-state space")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

savefig(fig, "07_layer_statistics.png")
plt.show()


## 9. Save arrays

Save the hidden-state batch and token metadata for later notebooks.

Notebook 13 can reload these outputs and compute curvature without re-running model inference.


In [ ]:
npz_path = ARR / "07_hidden_states.npz"

np.savez_compressed(
    npz_path,
    hidden=hidden,
    input_ids=input_ids,
    attention_mask=attention_mask,
    sentences=np.array(sentences, dtype=object),
)

print("saved:", npz_path)
print("hidden:", hidden.shape)


## 10. Compact interpretation

This notebook validates the first real object in the repo:

\[
x_i^\ell
\]

A sentence is not represented by one vector. It becomes a sequence of token-indexed hidden states across layers.

That sequence forms a trajectory.

Notebook 13 should now compute the paper's curvature metric from the saved hidden-state tensor.


## 11. Export and download

Package all Notebook 07 outputs.


In [ ]:
zip_path = RES / "07_outputs.zip"

files_to_zip = [
    FIG / "07_hidden_state_shapes.png",
    FIG / "07_token_counts.png",
    FIG / "07_sentence_trajectory_selected_layers.png",
    FIG / "07_layer_projection_grid.png",
    FIG / "07_layer_statistics.png",
    CSV / "07_hidden_state_shape.csv",
    CSV / "07_sentence_metadata.csv",
    CSV / "07_layer_statistics.csv",
    JS / "07_hidden_state_shape.json",
    JS / "07_layer_statistics.json",
    ARR / "07_hidden_states.npz",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

stats.head()


## Next notebook

**13 — Curvature**

The next notebook should compute the curvature metric from the saved hidden-state tensor.

Candidate outputs:

```text
13_curvature_by_layer.png
13_trained_profile_sentence_batch.png
13_curvature_heatmap.png
13_curvature_summary.csv
```

Core question:

```text
Do sentence trajectories become straighter from early to middle transformer layers?
```
